# 12 — 床反力から関節トルクへ

立脚のGRF変換と遊脚の追従制御が、同じ脚トルク出力へ合流する仕組みを確認します。

**前提**: `11_acados_and_receding_horizon.ipynb`

> 読み方: 「直感 → 数式 → 上流コード → 小実験 → 解釈」の順です。
> `実装事実` と書いた箇所は現行 `external/Quadruped-PyMPC` のコード、
> `学習用モデル` は理解のために単純化した再実装です。

In [1]:
from pathlib import Path
import os, sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebook_pympc":
    ROOT = ROOT.parent
PYMPC_ROOT = ROOT / "external" / "Quadruped-PyMPC"
assert PYMPC_ROOT.exists(), f"Quadruped-PyMPC が見つかりません: {PYMPC_ROOT}"
if str(PYMPC_ROOT) not in sys.path:
    sys.path.insert(0, str(PYMPC_ROOT))

os.environ.setdefault("ACADOS_SOURCE_DIR", str(PYMPC_ROOT / "quadruped_pympc" / "acados"))
os.environ.setdefault("MUJOCO_GL", "egl")
print("workspace :", ROOT)
print("PyMPC root:", PYMPC_ROOT)

workspace : /home/takuya/work/mpc_dog
PyMPC root: /home/takuya/work/mpc_dog/external/Quadruped-PyMPC


## 立脚脚

virtual workより

\[
\tau_{stance}=-J^TF^{cmd}
\]

符号は「ロボットが地面へ及ぼす力」と「地面がロボットへ及ぼす力」の定義で変わります。
現行実装の符号を正にしてください。

## 遊脚脚

Cartesian PDとfeedback linearizationを使い、足位置軌道を追従します。
contact scheduleが立脚・遊脚のどちらの式を使うかを選びます。

In [2]:
import numpy as np

# 1脚の並進Jacobian: foot_velocity_W = J_W(q) @ joint_velocity
J = np.array([[0.0, 0.20, 0.15],
              [0.18, 0.00, 0.00],
              [0.00, 0.12, 0.22]])  # [m/rad], shape (3 foot axes, 3 joints)

# MPC出力のworld座標GRF。ここでは前方10 N、上向き50 N。
F = np.array([10., 0., 50.])  # [N], shape (3,)

# 仮想仕事 deltaW = F.T*delta_p = F.T*J*delta_q よりJ.T@F。
# 上流は力の作用方向の定義に合わせて負号を付け tau_stance=-J.T@F とする。
tau = -J.T @ F  # [N m], shape (3,)
print("J shape:", J.shape, "F shape:", F.shape)
print("stance torque:", tau, "N m")
assert tau.shape == (3,)

J shape: (3, 3) F shape: (3,)
stance torque: [  0.   -8.  -12.5] N m


In [3]:
limits = np.array([23.7, 23.7, 45.4]) * 0.9
clipped = np.clip(tau, -limits, limits)
print("soft limits:", limits)
print("clipped    :", clipped)
print("saturated? :", np.abs(tau) > limits)

soft limits: [21.33 21.33 40.86]
clipped    : [  0.   -8.  -12.5]
saturated? : [False False False]


トルク飽和は原因ではなく結果です。大きいGRF、悪い足位置によるJacobian、
高い遊脚ゲイン、短いswing時間のいずれでも起きます。
飽和した脚・位相・寄与項を分解してから調整します。

## 章末チェック

出力を眺めるだけでなく、次を自分の言葉で答えてください。

1. この章の入力・出力の shape、単位、座標系は何か。
2. 変更可能な量と、他の章から渡される量は何か。
3. パラメータを2倍にしたとき、どのグラフがどちらへ変化するか。
4. 現行実装の事実と、学習用の近似を区別できるか。